In [5]:
from dotenv import load_dotenv
import json

load_dotenv()

True

In [6]:
from openai import OpenAI

client = OpenAI()

In [7]:
_COMPACT_PROMPT = """
당신은 요약 전문가입니다. 다음 요구사항을 충족시키는 요약문을 작성해주세요

반드시 보존할 것:
  - 사용자의 원래 요청과 현재 목표
  - 내려진 결정사항과 그 이유
  - 아직 해결되지 않은 문제
  - 사용자가 언급한 선호사항/제약조건

  버려도 되는 것:
  - 인사, 감사 등 사교적 발화
  - 이미 완료된 중간 과정의 세부사항
  - 반복된 내용
"""

In [13]:
def count_tokens(messages: list[dict]) -> int:
    """러프한 토큰 계산기 대충 모듈 3 으로 때린다"""
    return len(json.dumps(messages)) / 3

In [17]:
def compact_history(messages: list[dict], max_tokens: int = 4000) -> list[dict]:
    # if count_tokens(messages) <= max_tokens:
    #     return messages

    recent = messages[-4:]
    old = messages[:-4]

    input_messages = [
        {"role": "system", "content": _COMPACT_PROMPT},
        {"role": "user", "content": f"대화: {old}"}
    ]

    response = client.responses.create(
        model = "gpt-4.1",
        input = input_messages
    )

    return response.output
    

In [24]:
with open("D:\TEST\orchestrator_loop\\test_server\\compaction_test\\data\\english_version_01.json", "r") as f:
    data = json.load(f)

result = compact_history(data)

In [25]:
print(result[0].content[0].text)

요약문:

사용자는 FastAPI로 SSE 기반 스트리밍 서비스를 개발하며, 멀티리퀘스트 동시 처리 시 응답 순서가 어긋나는 문제를 해결하고자 함. 현 목표는 안정적인 응답 순서 보장 및 비용 효율적인 아키텍처 도입, 대용량 대화 관리, 번역 파이프라인의 자리표시자 보존, 오픈소스 모델의 효율적 운영 등이다.

결정사항 및 이유:
- SSE 응답 순서 보장을 위해 요청별 큐와 인덱싱 또는 하이브리드 메시지 저장(리스트+해시) 방식을 적용하기로 함(이유: 응답 순서 및 부분 수정 용이성 개선).
- 라우팅용 경량 모델(GPT-4.1-mini)과 추론용 고성능 모델(GPT-5-chat-latest)의 분리 운용이 비용 절감에 효과적이라 판단.
- 대화 히스토리 요약은 시스템 메시지로 삽입하고, 최근 메시지는 직접 요약하지 않는 점진적·계층적 요약 전략을 활용하여 정보 손실 최소화.
- 번역 파이프라인 자리표시자 보존은 출력 JSON 파싱 검증 및 구조화 출력 API 도입으로 관리하기로 결정(이유: 형식 오류 및 자리표시자 훼손 방지).
- vLLM 등 오픈소스 모델의 컨테이너 기동 지연 원인은 대용량 모델 다운로드와 이미지 풀로 인식.

아직 해결되지 않은 문제:
- 컨테이너 기동(특히 RunPod/vLLM) 속도 개선 및 최적화 방안 미정.
- 대화 컨텍스트 윈도우 초과시 요약 빈도와 정보 손실 최적 균형 미완.

사용자 선호사항/제약조건:
- 실시간 응답 순서 보장
- 부분 메시지 갱신 가능 저장 방식
- 대용량(수만 토큰) 대화 히스토리 처리
- 자리표시자 보존과 정확한 형식 보장
- 비용 효율적이면서 확장 가능한 아키텍처
- 오픈소스 모델 활용 및 기동 시간 단축
